In [1]:
import pandas as pd
import gpxpy
import os
import re

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [2]:
data_path = './input/AE_HO_007_2025-11-23_16_42_57_raw.csv'
track_path = './input/Equipage Anita Conti.gpx'
instrument_name = 'AE_HO_007' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"
csv_encoding = 'utf-8' # or 'latin-1' depending on your file encoding

## Étape 1 : Charger les données GPX

In [3]:
def load_gpx_tracks(file_path):
    with open(file_path, 'r') as gpx_file:
        gpx = gpxpy.parse(gpx_file)
    tracks = []
    for track in gpx.tracks:
        points = []
        for segment in track.segments:
            for point in segment.points:
                points.append({
                    'datetime': point.time,
                    'lat': point.latitude,
                    'long': point.longitude
                })
        if points:
            df_gpx = pd.DataFrame(points)
            df_gpx['datetime'] = pd.to_datetime(df_gpx['datetime']).dt.tz_convert('UTC')
            df_gpx = df_gpx.set_index('datetime')
            tracks.append(df_gpx)
    return tracks

gpx_tracks = load_gpx_tracks(track_path)

# Afficher la plage de temps de chaque tracée
for i, df_gpx in enumerate(gpx_tracks):
    print(f"Tracée {i+1} : de {df_gpx.index.min()} à {df_gpx.index.max()}")

Tracée 1 : de 2025-09-19 08:45:33+00:00 à 2025-09-20 15:36:13+00:00


## Étape 2 : Charger les données CSV

In [13]:
def rename_columns_with_regex(df):
    # Mapping with regex patterns (the number is replaced by \\d+)
    col_map = {
        r'Température.*': 'sea_temp',
        r'Pression absolue.*': 'sea_pres',
        r'Conductivité électrique.*': 'ec_abs',
        r'Conductivité spécifique.*': 'ec25',
        r'Salinité.*': 'sea_sal',
        r'Matières solides dissoutes totales.*': 'total_dissolved_solids',
    }
    new_columns = {}
    for col in df.columns:
        new_col = col
        for pattern, replacement in col_map.items():
            if re.fullmatch(pattern, col):
                new_col = replacement
                break
        new_columns[col] = new_col
    return df.rename(columns=new_columns)

df = pd.read_csv(data_path, low_memory=False, parse_dates=[1], date_format="%m/%d/%Y %H:%M:%S", delimiter=';', encoding=csv_encoding)

# Renommer les colonnes avec regex
# La colonne 'Date et heure (CEST)' reste renommée explicitement
if 'Date et heure (CEST)' in df.columns:
    df = df.rename(columns={'Date et heure (CEST)': 'datetime'})
df = rename_columns_with_regex(df)

# Supprimer les colonnes inutiles
df = df.drop(
    columns=['#', 'Hôte connecté', 'Fin du fichier', 'Démarré', 'Arrêté', 'Bouton Haut', 'Bouton Bas', 'Détection d’eau'],
    errors='ignore'
)

print(df.head())

# Convertir le fuseau horaire de CEST en UTC
df['datetime'] = df['datetime'].dt.tz_localize('Europe/Paris')
df['datetime'] = df['datetime'].dt.tz_convert('UTC')
df = df.set_index('datetime')

             datetime  sea_temp  sea_pres   ec_abs     ec25  sea_sal  \
0 2025-07-21 18:43:27       NaN       NaN      NaN      NaN      NaN   
1 2025-07-21 18:43:31     11.42   102.400  35789.3  50063.0    31.41   
2 2025-07-21 18:43:32       NaN       NaN      NaN      NaN      NaN   
3 2025-07-21 18:44:31     11.39   102.398  35749.2  50056.0    31.40   
4 2025-07-21 18:45:31     11.36   102.397  35736.1  50086.9    31.41   

   total_dissolved_solids  
0                     NaN  
1                23263.06  
2                     NaN  
3                23236.96  
4                23228.45  


In [14]:
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())

Plage de dates CSV : 2025-07-21 16:43:27+00:00 à 2025-11-23 16:40:50+00:00


## Étape 3 : Interpolation de la position des mesures à partir des traces GPS  

In [15]:
def interpolate_position(row, df_gpx):
    t = row.name
    idx = df_gpx.index.searchsorted(t, side='right') - 1
    if idx < 0 or idx >= len(df_gpx) - 1:
        return None, None  # Hors des limites
    point_A = df_gpx.iloc[idx]
    point_B = df_gpx.iloc[idx + 1]
    tA, tB = point_A.name, point_B.name
    latA, longA = point_A['lat'], point_A['long']
    latB, longB = point_B['lat'], point_B['long']
    ratio = (t - tA).total_seconds() / (tB - tA).total_seconds()
    lat = latA + (latB - latA) * ratio
    long = longA + (longB - longA) * ratio
    return lat, long

def interpolate_all_tracks(df, gpx_tracks):
    dfs = []
    for df_gpx in gpx_tracks:
        # Filtrer les mesures dans la plage de la tracée
        mask = (df.index >= df_gpx.index.min()) & (df.index <= df_gpx.index.max())
        df_sub = df[mask].copy()
        if df_sub.empty:
            continue
        df_sub[['lat', 'long']] = df_sub.apply(lambda row: interpolate_position(row, df_gpx), axis=1, result_type='expand')
        df_sub = df_sub.dropna(subset=['lat', 'long'])
        dfs.append(df_sub)
    if dfs:
        return pd.concat(dfs).sort_index()
    else:
        return pd.DataFrame()

In [16]:
df_interp = interpolate_all_tracks(df, gpx_tracks)

## Étape 4 : Exporter les données

In [ ]:
os.makedirs(output_dir, exist_ok=True)

if not df_interp.empty:
    start_at = df_interp.index.min().strftime('%Y-%m-%d_%H_%M_%S')
    output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}_with_positions.csv")
    df_interp.to_csv(output_csv_final, sep=';', date_format='%Y-%m-%dT%H:%M:%SZ')
    print(f"Fichier exporté : {output_csv_final}")
else:
    print("Aucune donnée interpolée à exporter.")

Fichier exporté : output/AE_HO_007_2025-09-19_08_46_01_with_positions.csv
